## Start here
Select the project `.venv` Python kernel in VS Code, then **Restart Kernel**.
Run the Java setup cell first, then the Spark startup cell below. After both succeed, continue to the optional database configuration cells.
If Spark startup fails, restart the kernel before retrying; changing `JAVA_HOME` cannot replace a JVM that is already running.


In [1]:
# Select this project's .venv kernel in VS Code.
# After changing Java, restart the kernel and run this cell before Spark.
import os
from pathlib import Path

java_home = Path("/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home")
if not (java_home / "bin" / "java").is_file():
    raise FileNotFoundError(f"Java 17 was not found at {java_home}")

os.environ["JAVA_HOME"] = str(java_home)
os.environ["PATH"] = str(java_home / "bin") + os.pathsep + os.environ.get("PATH", "")

# An existing Spark JVM keeps its original Java version until kernel restart.
import sys

pyspark_module = sys.modules.get("pyspark")
if pyspark_module is not None:
    gateway = pyspark_module.SparkContext._gateway
    if gateway is not None:
        java_version = gateway.jvm.java.lang.System.getProperty("java.specification.version")
        if java_version != "17":
            raise RuntimeError("Spark already has a different Java runtime. Restart Kernel, then run from the first cell.")


### Start Spark (one session)
`getOrCreate()` reuses the session when this cell is rerun successfully.

In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("local[1]")
    .appName("vijay-traffic-exploration")
    .config("spark.jars.packages", "org.postgresql:postgresql:42.5.0")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)

print("Spark started successfully")

print("Java runtime:", spark.sparkContext._jvm.java.lang.System.getProperty("java.version"))
print("Spark version:", spark.version)


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/13 20:50:16 WARN Utils: Your hostname, Zipcoders-MacBook-6.local, resolves to a loopback address: 127.0.0.1; using 192.168.88.125 instead (on interface en0)
26/09/13 20:50:16 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/Users/vijayarajan/Projects/SparkCity_Capstone/.venv/lib/python3.13/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/vijayarajan/.ivy2.5.2/cache
The jars for the packages stored in: /Users/vijayarajan/.ivy2.5.2/jars
org.postgresql#postgresql added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-38dd02f1-3a18-49ff-a5e5-13bb6291c8cc;1.0
	confs: [default]
	found org.postgresql#postgresql;42.5.0 in central
	found org.checkerframework#checker-qual;3.5.0 in central
:: resolution report :: resolve 90ms :: artifacts dl 2ms
	:: 

Spark started successfully
Java runtime: 17.0.20.1
Spark version: 4.2.0


### Optional database configuration imports
The following cells load connection settings only; they do not query or write to the database.

In [3]:
from dotenv import load_dotenv
from psycopg.conninfo import conninfo_to_dict

from sparkcityx.database import get_database_url


### Load database configuration

In [4]:
load_dotenv("../secrets/.env")

info = conninfo_to_dict(get_database_url())

print("Database configuration loaded successfully")

Database configuration loaded successfully


### Read Traffic: PostgreSQL → JDBC → Spark DataFrame
JDBC is the connector Spark uses to read `sparkcity.traffic_sensors` from PostgreSQL. Reuse the connection settings above, keeping credentials in properties instead of the JDBC URL. This cell only reads data; Spark fetches rows when we explore the DataFrame below.


In [5]:
from urllib.parse import quote

# Keep credentials separate from the URL, and preserve the SSL settings.
jdbc_host = info["host"]
if ":" in jdbc_host and not jdbc_host.startswith("["):
    jdbc_host = f"[{jdbc_host}]"  # IPv6 host address
jdbc_url = (
    f"jdbc:postgresql://{jdbc_host}:{info.get('port', '5432')}/"
    f"{quote(info['dbname'], safe='')}"
)
jdbc_properties = {
    "user": info["user"],
    "password": info["password"],
    "driver": "org.postgresql.Driver",
    "readOnly": "true",
    "readOnlyMode": "always",
}
for option in ("sslmode", "sslrootcert", "sslcert", "sslkey"):
    if option in info:
        jdbc_properties[option] = info[option]

# Do not display the connection URL or properties.
try:
    traffic_df = spark.read.jdbc(
        url=jdbc_url,
        table="sparkcity.traffic_sensors",
        properties=jdbc_properties,
    )
except Exception:
    raise RuntimeError("Traffic read failed. Check database access and connection settings privately.") from None


### Inspect the Traffic DataFrame
Print the column names and types, count the rows (expected: **36,000**), and preview five rows without truncation. These are read-only operations. The preview has no guaranteed order.


In [6]:
traffic_df.printSchema()
try:
    traffic_row_count = traffic_df.count()
    print(f"Traffic row count: {traffic_row_count:,} (expected: 36,000)")
    traffic_df.show(5, truncate=False)
except Exception:
    raise RuntimeError("Traffic inspection failed. Check database access privately.") from None


root
 |-- sensor_id: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- location_lat: double (nullable = true)
 |-- location_lon: double (nullable = true)
 |-- vehicle_count: integer (nullable = true)
 |-- avg_speed: double (nullable = true)
 |-- congestion_level: string (nullable = true)
 |-- road_type: string (nullable = true)



Traffic row count: 36,000 (expected: 36,000)
+---------+-------------------+------------+------------+-------------+---------+----------------+-----------+
|sensor_id|timestamp          |location_lat|location_lon|vehicle_count|avg_speed|congestion_level|road_type  |
+---------+-------------------+------------+------------+-------------+---------+----------------+-----------+
|TRF-0001 |2025-01-01 00:00:00|40.680561   |-74.0492    |109          |16.17    |high            |arterial   |
|TRF-0002 |2025-01-01 00:05:00|40.680511   |-74.049047  |81           |31.34    |medium          |highway    |
|TRF-0003 |2025-01-01 00:10:00|40.680129   |-74.047605  |55           |15.2     |medium          |residential|
|TRF-0004 |2025-01-01 00:15:00|40.680509   |-74.047168  |46           |33.1     |low             |arterial   |
|TRF-0005 |2025-01-01 00:20:00|40.680773   |-74.045866  |42           |18.56    |low             |residential|
+---------+-------------------+------------+------------+----------

## Traffic Data Quality Validation
Pass the existing Spark DataFrame `traffic_df` to the team's reusable validator. It checks the Traffic rules without changing the data or writing to PostgreSQL.

The report shows whether the dataset passes those rules, its record count, and any problems. Empty lists or dictionaries mean no issues were reported for that check; the validator lists only nonzero null and violation counts. Duplicates are extra rows sharing the same `sensor_id` and `timestamp`. Timestamp checks test whether values can be parsed, not whether they fall within a particular date range.

Numeric summaries include count, mean, standard deviation, minimum, quartiles, and maximum. We display the complete returned report below, including any failures.


In [7]:
from sparkcityx.data_quality import validate_dataframe

traffic_validation = validate_dataframe(traffic_df, "traffic")


26/09/13 20:50:36 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [8]:
from pprint import pprint

print("Traffic validation passed:", traffic_validation["valid"])
print("Record count:", traffic_validation["record_count"])

# Display every other result, including problems and numeric summaries.
for check, result in traffic_validation.items():
    if check not in ("valid", "record_count"):
        print("\n" + check.replace("_", " ").capitalize() + ":")
        pprint(result, sort_dicts=False, width=100)


Traffic validation passed: True
Record count: 36000

Dataset type:
'traffic'

Missing columns:
[]

Non numeric columns:
[]

Null counts:
{}

Duplicate count:
0

Range violations:
{}

Value violations:
{}

Timestamp violations:
{}

Cross field violations:
{}

Numeric summary:
[{'summary': 'count',
  'location_lat': '36000',
  'location_lon': '36000',
  'vehicle_count': '36000',
  'avg_speed': '36000'},
 {'summary': 'mean',
  'location_lat': '40.759780166416846',
  'location_lon': '-73.96510990627756',
  'vehicle_count': '76.39902777777777',
  'avg_speed': '18.92764888888897'},
 {'summary': 'stddev',
  'location_lat': '0.04606195685715278',
  'location_lon': '0.04907745793114205',
  'vehicle_count': '36.3166926220402',
  'avg_speed': '12.33655607918761'},
 {'summary': 'min',
  'location_lat': '40.680006',
  'location_lon': '-74.049999',
  'vehicle_count': '15',
  'avg_speed': '5.0'},
 {'summary': '25%',
  'location_lat': '40.719898',
  'location_lon': '-74.007667',
  'vehicle_count': '47

## Hourly Traffic Patterns
Use the existing `traffic_df` to summarize observations across all dates and sensors by hour of day. `withColumn()` adds an hour column to a new DataFrame, leaving `traffic_df` unchanged. `hour()` extracts an integer from 0 to 23 using Spark's session time zone.

`groupBy()` collects observations with the same hour. `agg()` calculates each group's observation count and averages; `avg()` computes an arithmetic mean. `orderBy()` puts the hours in chronological order. Average vehicle count is per observation, not a total number of vehicles for the hour.


In [9]:
from pyspark.sql import functions as F

hourly_traffic = (
    traffic_df
    .withColumn("hour", F.hour("timestamp"))
    .groupBy("hour")
    .agg(
        F.count("*").alias("observation_count"),
        F.avg("vehicle_count").alias("avg_vehicle_count"),
        F.avg("avg_speed").alias("avg_speed"),
    )
    .orderBy("hour")
)

print("Hour uses Spark session time zone:", spark.conf.get("spark.sql.session.timeZone"))
hourly_traffic.show(24, truncate=False)


Hour uses Spark session time zone: America/New_York


+----+-----------------+------------------+------------------+
|hour|observation_count|avg_vehicle_count |avg_speed         |
+----+-----------------+------------------+------------------+
|0   |1500             |69.52133333333333 |19.729246666666665|
|1   |1500             |68.15933333333334 |19.94379333333334 |
|2   |1488             |68.19825268817205 |20.213534946236578|
|3   |1512             |68.20833333333333 |19.469927248677255|
|4   |1500             |68.82             |19.582566666666693|
|5   |1500             |66.65733333333333 |19.772413333333343|
|6   |1500             |68.08333333333333 |20.025419999999993|
|7   |1500             |104.25333333333333|16.29609333333334 |
|8   |1500             |105.98066666666666|15.964066666666657|
|9   |1500             |69.00666666666666 |20.041513333333345|
|10  |1500             |68.844            |19.99206666666667 |
|11  |1500             |68.98866666666666 |19.691973333333326|
|12  |1500             |69.62666666666667 |19.286213333

### Highest and lowest hourly averages
Sort the hourly summary by each average and take the first row. Descending order finds the highest value; ascending order finds the lowest. If averages tie, the earlier hour is selected. These compare hourly averages across the whole dataset.


In [10]:
highest_traffic_hour = hourly_traffic.orderBy(F.desc("avg_vehicle_count"), "hour").first()
lowest_traffic_hour = hourly_traffic.orderBy(F.asc("avg_vehicle_count"), "hour").first()
highest_speed_hour = hourly_traffic.orderBy(F.desc("avg_speed"), "hour").first()
lowest_speed_hour = hourly_traffic.orderBy(F.asc("avg_speed"), "hour").first()

print(f"Highest average vehicle count: {highest_traffic_hour['hour']:02d}:00 — {highest_traffic_hour['avg_vehicle_count']:.2f}")
print(f"Lowest average vehicle count: {lowest_traffic_hour['hour']:02d}:00 — {lowest_traffic_hour['avg_vehicle_count']:.2f}")
print(f"Highest average speed: {highest_speed_hour['hour']:02d}:00 — {highest_speed_hour['avg_speed']:.2f}")
print(f"Lowest average speed: {lowest_speed_hour['hour']:02d}:00 — {lowest_speed_hour['avg_speed']:.2f}")


Highest average vehicle count: 17:00 — 107.60
Lowest average vehicle count: 05:00 — 66.66
Highest average speed: 15:00 — 20.38
Lowest average speed: 16:00 — 15.31


### Interpretation of the observed results
In this run, hours use Spark's `America/New_York` session time zone. The highest average vehicle count is at **17:00 (107.60)** and the lowest is at **05:00 (66.66)**. The highest average speed is at **15:00 (20.38)** and the lowest is at **16:00 (15.31)**, in the source data's speed units.

Average vehicle counts rise to about **104–106 at 07:00–08:00** and **107–108 at 16:00–17:00**, remaining high at **18:00 (106.16)**. These hours have lower average speeds (about **15–16**) than most other hours (about **19–20**). This is a morning and late-afternoon peak pattern in the dataset; the summary alone does not establish its cause.

All 24 hours are present, accounting for **36,000 observations**. Most hours have **1,500 observations**; hour 02 has **1,488** and hour 03 has **1,512**. These are averages across dates and sensors, not the traffic profile of one specific day or road.


## City Zones Exploration
City Zones is reference/geographic data. Each zone has latitude and longitude bounds that will later help map Traffic sensor coordinates to zones. For now, we only read and inspect the zones, using the existing SparkSession and secure JDBC settings; no Traffic mapping is performed.


In [11]:
try:
    city_zones_df = spark.read.jdbc(
        url=jdbc_url,
        table="sparkcity.city_zones",
        properties=jdbc_properties,
    )
except Exception:
    raise RuntimeError("City Zones read failed. Check database access privately.") from None


### Inspect the reference data
Print the schema, count the rows and distinct zone IDs, and preview ten rows ordered by zone ID. `distinct()` removes repeated values from the selected column so we can see the different zone types.


In [12]:
city_zones_df.printSchema()
city_zones_row_count = city_zones_df.count()
city_zones_distinct_ids = city_zones_df.select("zone_id").distinct().count()
print("City Zones row count:", city_zones_row_count)
print("Distinct zone_id count:", city_zones_distinct_ids)
city_zones_df.orderBy("zone_id").show(10, truncate=False)
print("Distinct zone types:")
city_zones_df.select("zone_type").distinct().orderBy("zone_type").show(truncate=False)


root
 |-- zone_id: string (nullable = true)
 |-- zone_name: string (nullable = true)
 |-- zone_type: string (nullable = true)
 |-- lat_min: double (nullable = true)
 |-- lat_max: double (nullable = true)
 |-- lon_min: double (nullable = true)
 |-- lon_max: double (nullable = true)
 |-- population: integer (nullable = true)



City Zones row count: 36000
Distinct zone_id count: 36000


+---------+---------------------+-----------+-------+---------+----------+----------+----------+
|zone_id  |zone_name            |zone_type  |lat_min|lat_max  |lon_min   |lon_max   |population|
+---------+---------------------+-----------+-------+---------+----------+----------+----------+
|ZONE-0001|Residential Zone 0001|residential|40.68  |40.680842|-74.05    |-74.049105|3824      |
|ZONE-0002|Commercial Zone 0002 |commercial |40.68  |40.680842|-74.049105|-74.048211|602       |
|ZONE-0003|Industrial Zone 0003 |industrial |40.68  |40.680842|-74.048211|-74.047316|663       |
|ZONE-0004|Mixed Use Zone 0004  |mixed_use  |40.68  |40.680842|-74.047316|-74.046421|3506      |
|ZONE-0005|Park Zone 0005       |park       |40.68  |40.680842|-74.046421|-74.045526|57        |
|ZONE-0006|Campus Zone 0006     |campus     |40.68  |40.680842|-74.045526|-74.044632|1943      |
|ZONE-0007|Residential Zone 0007|residential|40.68  |40.680842|-74.044632|-74.043737|3679      |
|ZONE-0008|Commercial Zone 000

+-----------+
|zone_type  |
+-----------+
|campus     |
|commercial |
|industrial |
|mixed_use  |
|park       |
|residential|
+-----------+



### Validate with the team's existing rules
Reuse `validate_dataframe()` to check required columns, nulls, duplicate IDs, numeric ranges, and boundary ordering. Empty issue lists or dictionaries mean no problems were reported for that check. The existing `lat_bounds` and `lon_bounds` rules check `lat_min <= lat_max` and `lon_min <= lon_max`; we display their results instead of implementing another validator.


In [13]:
from sparkcityx.data_quality import validate_dataframe
from pprint import pprint

city_zones_validation = validate_dataframe(city_zones_df, "city_zones")
print("City Zones validation passed:", city_zones_validation["valid"])
for check, result in city_zones_validation.items():
    if check != "valid":
        print("\n" + check.replace("_", " ").capitalize() + ":")
        pprint(result, sort_dicts=False, width=100)

print("\nRows with lat_min > lat_max:", city_zones_validation["cross_field_violations"].get("lat_bounds", 0))
print("Rows with lon_min > lon_max:", city_zones_validation["cross_field_violations"].get("lon_bounds", 0))


City Zones validation passed: True

Dataset type:
'city_zones'

Record count:
36000

Missing columns:
[]

Non numeric columns:
[]

Null counts:
{}

Duplicate count:
0

Range violations:
{}

Value violations:
{}

Timestamp violations:
{}

Cross field violations:
{}

Numeric summary:
[{'summary': 'count',
  'lat_min': '36000',
  'lat_max': '36000',
  'lon_min': '36000',
  'lon_max': '36000',
  'population': '36000'},
 {'summary': 'mean',
  'lat_min': '40.75935789444529',
  'lat_max': '40.760200000001056',
  'lon_min': '-73.96555921051029',
  'lon_max': '-73.96466447367668',
  'population': '3022.003583333333'},
 {'summary': 'stddev',
  'lat_min': '0.046061039194908414',
  'lat_max': '0.04606103970160988',
  'lon_min': '0.04907805063506496',
  'lon_max': '0.04907805063595126',
  'population': '2855.8303511974095'},
 {'summary': 'min',
  'lat_min': '40.68',
  'lat_max': '40.680842',
  'lon_min': '-74.05',
  'lon_max': '-74.049105',
  'population': '0'},
 {'summary': '25%',
  'lat_min': '40

### Inspect geographic extent
`min()` and `max()` find the outermost boundaries across all zones. This overall extent does not prove that zones cover every point inside it or that zones do not overlap. Later mapping will need to consider individual rectangles and how to handle points on shared edges.


In [14]:
city_zones_bounds = city_zones_df.agg(
    F.min("lat_min").alias("minimum_lat_min"),
    F.max("lat_max").alias("maximum_lat_max"),
    F.min("lon_min").alias("minimum_lon_min"),
    F.max("lon_max").alias("maximum_lon_max"),
)
city_zones_bounds.show(truncate=False)


+---------------+---------------+---------------+---------------+
|minimum_lat_min|maximum_lat_max|minimum_lon_min|maximum_lon_max|
+---------------+---------------+---------------+---------------+
|40.68          |40.84          |-74.05         |-73.88         |
+---------------+---------------+---------------+---------------+



### What the actual City Zones data tells us
The table contains **36,000 rows and 36,000 distinct zone IDs**. Its six zone types are **campus, commercial, industrial, mixed_use, park, and residential**. The team's validator passed: no missing required columns, nulls, duplicate IDs, numeric-type issues, range violations, or reversed latitude/longitude bounds were reported. All rows satisfy `lat_min <= lat_max` and `lon_min <= lon_max`.

The overall geographic extent is **40.68 to 40.84 latitude** and **-74.05 to -73.88 longitude**. The previously observed Traffic coordinate ranges fall inside this overall extent, but this alone does not prove that each Traffic point matches a zone.

The preview shows small rectangles with shared edges (for example, ZONE-0001's `lon_max` equals ZONE-0002's `lon_min`). A future mapping must define how to handle points exactly on shared edges, since inclusive comparisons could produce multiple matches. Unique IDs and valid bounds do not prove the absence of geographic overlaps or gaps; those have not been tested here. With 36,000 zones, this is a sizeable reference table rather than a handful of broad districts.

The shared City Zones rules do not define allowed zone types or timestamp checks, so empty results for those categories do not represent additional tests. No Traffic-to-Zone mapping has been performed.


## Traffic to City Zone Mapping
Traffic contains point coordinates (`location_lat`, `location_lon`); City Zones contains rectangles (`lat_min`, `lat_max`, `lon_min`, `lon_max`). A non-equi/range join compares coordinates with boundary ranges instead of simply matching IDs. We start with inclusive boundaries and retain every candidate, without choosing a final zone.

Aliases `t` and `z` make it clear which DataFrame supplies each column. The join condition requires both latitude and longitude to fall inside the rectangle. To narrow the search, temporary 0.01-degree latitude buckets supply an equality join key: each rectangle is placed in every bucket it touches. The exact geographic conditions still decide matches, including shared edges. This avoids materializing the full Cartesian product; no full dataset is collected to Python or explicitly broadcast.


In [15]:
from pyspark.sql import functions as F

traffic_points = traffic_df.withColumn("_lat_bucket", F.floor(F.col("location_lat") * 100)).alias("t")
zone_rectangles = city_zones_df.withColumn(
    "_lat_bucket",
    F.explode(F.sequence(F.floor(F.col("lat_min") * 100), F.floor(F.col("lat_max") * 100))),
).alias("z")

traffic_zone_candidates = (
    traffic_points.join(
        zone_rectangles,
        (F.col("t._lat_bucket") == F.col("z._lat_bucket"))
        & F.col("t.location_lat").between(F.col("z.lat_min"), F.col("z.lat_max"))
        & F.col("t.location_lon").between(F.col("z.lon_min"), F.col("z.lon_max")),
        "left",
    )
    .select("t.sensor_id", "t.timestamp", "t.location_lat", "t.location_lon",
            "z.zone_id", "z.zone_name", "z.lat_min", "z.lat_max", "z.lon_min", "z.lon_max")
    .withColumn("strict_interior", (F.col("location_lat") > F.col("lat_min"))
                & (F.col("location_lat") < F.col("lat_max"))
                & (F.col("location_lon") > F.col("lon_min"))
                & (F.col("location_lon") < F.col("lon_max")))
    .cache()
)


### Count matches before enrichment
A spatial join can produce several rows for one Traffic record if several rectangles contain its point. A left join also keeps unmatched Traffic records with a null zone ID. `groupBy()` gathers candidates by the logical key (`sensor_id`, `timestamp`), and `count("zone_id")` counts only non-null zone IDs. Thus zero matches remain visible. We verify key uniqueness before interpreting these counts.

Strict-interior counts exclude all four rectangle edges only as a diagnostic. They do not establish a final assignment rule. Multiple strict-interior matches indicate overlapping rectangle interiors; multiple inclusive matches with at most one interior match depend on boundary inclusion.


In [16]:
original_traffic_records = traffic_df.count()
assert traffic_df.select("sensor_id", "timestamp").distinct().count() == original_traffic_records, "Traffic logical keys must be unique"
assert city_zones_df.select("zone_id").distinct().count() == city_zones_df.count(), "Zone IDs must be unique"

traffic_zone_match_counts = traffic_zone_candidates.groupBy("sensor_id", "timestamp").agg(
    F.count("zone_id").alias("zone_match_count"),
    F.sum(F.when(F.col("strict_interior"), 1).otherwise(0)).alias("interior_match_count"),
).cache()

mapping_quality = traffic_zone_match_counts.agg(
    F.count("*").alias("traffic_records"),
    F.sum(F.when(F.col("zone_match_count") == 1, 1).otherwise(0)).alias("exactly_one"),
    F.sum(F.when(F.col("zone_match_count") == 0, 1).otherwise(0)).alias("zero"),
    F.sum(F.when(F.col("zone_match_count") > 1, 1).otherwise(0)).alias("multiple"),
    F.max("zone_match_count").alias("maximum_matches"),
    F.sum(F.when((F.col("zone_match_count") > 1) & (F.col("interior_match_count") <= 1), 1).otherwise(0)).alias("boundary_dependent_multiple"),
    F.sum(F.when(F.col("interior_match_count") > 1, 1).otherwise(0)).alias("multiple_strict_interiors"),
    F.sum(F.when(F.col("zone_match_count") > F.col("interior_match_count"), 1).otherwise(0)).alias("records_touching_edges"),
)
print("Original Traffic records:", original_traffic_records)
mapping_quality.show(truncate=False)
assert traffic_zone_match_counts.count() == original_traffic_records

print("Unmatched sample (up to 5 records):")
traffic_zone_candidates.filter(F.col("zone_id").isNull()).select(
    "sensor_id", "timestamp", "location_lat", "location_lon"
).orderBy("sensor_id", "timestamp").show(5, truncate=False)

print("Multiple-match sample (all candidates for up to 5 logical keys):")
multiple_sample_keys = traffic_zone_match_counts.filter(F.col("zone_match_count") > 1).orderBy("sensor_id", "timestamp").limit(5)
multiple_sample = traffic_zone_candidates.join(multiple_sample_keys, ["sensor_id", "timestamp"], "inner")
multiple_sample.orderBy("sensor_id", "timestamp", "zone_id").show(50, truncate=False)

# Release diagnostic caches after displaying results; source DataFrames are unchanged.
traffic_zone_match_counts.unpersist()
traffic_zone_candidates.unpersist()


Original Traffic records: 36000


+---------------+-----------+----+--------+---------------+---------------------------+-------------------------+----------------------+
|traffic_records|exactly_one|zero|multiple|maximum_matches|boundary_dependent_multiple|multiple_strict_interiors|records_touching_edges|
+---------------+-----------+----+--------+---------------+---------------------------+-------------------------+----------------------+
|36000          |35921      |0   |79      |2              |79                         |0                        |79                    |
+---------------+-----------+----+--------+---------------+---------------------------+-------------------------+----------------------+



Unmatched sample (up to 5 records):
+---------+---------+------------+------------+
|sensor_id|timestamp|location_lat|location_lon|
+---------+---------+------------+------------+
+---------+---------+------------+------------+

Multiple-match sample (all candidates for up to 5 logical keys):
+---------+-------------------+------------+------------+----------+----------------------+---------+---------+----------+----------+---------------+----------------+--------------------+
|sensor_id|timestamp          |location_lat|location_lon|zone_id   |zone_name             |lat_min  |lat_max  |lon_min   |lon_max   |strict_interior|zone_match_count|interior_match_count|
+---------+-------------------+------------+------------+----------+----------------------+---------+---------+----------+----------+---------------+----------------+--------------------+
|TRF-0019 |2025-01-21 21:30:00|40.706845   |-73.934579  |ZONE-6019 |Residential Zone 6019 |40.706105|40.706947|-73.935474|-73.934579|false    

DataFrame[sensor_id: string, timestamp: timestamp, location_lat: double, location_lon: double, zone_id: string, zone_name: string, lat_min: double, lat_max: double, lon_min: double, lon_max: double, strict_interior: boolean]

### Mapping diagnosis from the actual results
Of **36,000 original Traffic records**, **35,921 match exactly one zone**, **0 match zero zones**, and **79 match multiple zones**. The maximum is **2 zones per record**. Logical Traffic keys and zone IDs were checked for uniqueness before counting.

All **79 multiple-match records depend on inclusive boundaries**: excluding edges leaves at most one strict-interior match for each. **No Traffic point matched multiple strict interiors**, and **79 records touch zone edges**. The displayed samples lie on shared latitude or longitude edges and have zero strict-interior matches. For example, TRF-0019 at 2025-01-21 21:30:00 has longitude -73.934579, equal to ZONE-6019's upper longitude and ZONE-6020's lower longitude.

The inclusive join therefore produces **36,079 candidate rows**, not 36,000 uniquely enriched records. Simply excluding every boundary is not a complete solution: the sampled edge points would become unmatched. We have retained all candidates and have not chosen a final assignment rule.

These results establish coverage for the observed Traffic points under inclusive comparisons. They do not prove that every possible point in the geographic extent is covered or that zone interiors never overlap elsewhere. Comparisons use the stored coordinate precision; no tolerance or rounding rule was introduced.


## Final Traffic Zone Enrichment
The inclusive join produced **36,079 candidate rows from 36,000 Traffic records** because 79 boundary records each matched two zones. Keeping both assignments would double-count those observations in later analysis; dropping boundary records would lose valid observations.

For this dataset, keep a single candidate as-is and resolve the diagnosed shared-boundary cases by choosing the **smallest zone_id in ascending string sort order**. This is a deterministic technical tie-breaker, not a claim that one zone is geographically more correct.

A `Window` defines a group and ordering for calculations without collapsing its rows. `partitionBy("sensor_id", "timestamp")` groups candidates for one Traffic record. `orderBy("zone_id")` places them in a repeatable order. `row_number()` numbers each candidate within that group; keeping row 1 gives one assignment per logical key and prevents row multiplication. We then attach the assignment to the original Traffic columns and add the zone name and type.


In [17]:
from pyspark.sql import Window

traffic_key = ["sensor_id", "timestamp"]
zone_choice_window = Window.partitionBy(*traffic_key).orderBy(F.col("zone_id").asc())

selected_traffic_zones = (
    traffic_zone_candidates
    .withColumn("zone_rank", F.row_number().over(zone_choice_window))
    .filter(F.col("zone_rank") == 1)
    .select(*traffic_key, "zone_id")
)

traffic_enriched_df = (
    traffic_df.join(selected_traffic_zones, traffic_key, "left")
    .join(city_zones_df.select("zone_id", "zone_name", "zone_type"), "zone_id", "left")
    .select(*traffic_df.columns, "zone_id", "zone_name", "zone_type")
)


### Verify every original record is preserved
Counts alone cannot prove that the correct records survived. We also compare logical keys in both directions and compare the original Traffic columns, including duplicate occurrences, using `exceptAll()`. These checks only inspect DataFrames; they do not alter source data or write output.


In [18]:
enriched_row_count = traffic_enriched_df.count()
enriched_key_count = traffic_enriched_df.select(*traffic_key).distinct().count()
null_zone_assignments = traffic_enriched_df.filter(F.col("zone_id").isNull()).count()
duplicate_traffic_keys = traffic_enriched_df.groupBy(*traffic_key).count().filter(F.col("count") > 1).count()
missing_traffic_keys = traffic_df.select(*traffic_key).join(traffic_enriched_df.select(*traffic_key), traffic_key, "left_anti").count()
extra_traffic_keys = traffic_enriched_df.select(*traffic_key).join(traffic_df.select(*traffic_key), traffic_key, "left_anti").count()
original_columns_preserved = (
    traffic_df.exceptAll(traffic_enriched_df.select(*traffic_df.columns)).limit(1).count() == 0
    and traffic_enriched_df.select(*traffic_df.columns).exceptAll(traffic_df).limit(1).count() == 0
)

print("Final enriched rows:", enriched_row_count)
print("Distinct Traffic logical keys:", enriched_key_count)
print("Null zone assignments:", null_zone_assignments)
print("Remaining duplicate Traffic keys:", duplicate_traffic_keys)
print("Missing / extra Traffic keys:", missing_traffic_keys, "/", extra_traffic_keys)
print("Original Traffic column values preserved:", original_columns_preserved)

assert enriched_row_count == traffic_df.count() == 36000
assert enriched_key_count == 36000
assert null_zone_assignments == duplicate_traffic_keys == missing_traffic_keys == extra_traffic_keys == 0
assert original_columns_preserved

traffic_enriched_df.orderBy(*traffic_key).show(5, truncate=False)


Final enriched rows: 36000
Distinct Traffic logical keys: 36000
Null zone assignments: 0
Remaining duplicate Traffic keys: 0
Missing / extra Traffic keys: 0 / 0
Original Traffic column values preserved: True


+---------+-------------------+------------+------------+-------------+---------+----------------+-----------+----------+----------------------+-----------+
|sensor_id|timestamp          |location_lat|location_lon|vehicle_count|avg_speed|congestion_level|road_type  |zone_id   |zone_name             |zone_type  |
+---------+-------------------+------------+------------+-------------+---------+----------------+-----------+----------+----------------------+-----------+
|TRF-0001 |2025-01-01 00:00:00|40.680561   |-74.0492    |109          |16.17    |high            |arterial   |ZONE-0001 |Residential Zone 0001 |residential|
|TRF-0001 |2025-01-11 10:00:00|40.693059   |-73.914918  |66           |37.93    |medium          |highway    |ZONE-3001 |Residential Zone 3001 |residential|
|TRF-0001 |2025-01-21 20:00:00|40.706654   |-73.951563  |101          |21.94    |high            |highway    |ZONE-6001 |Residential Zone 6001 |residential|
|TRF-0001 |2025-02-01 06:00:00|40.719784   |-73.98719   |4

### Inspect the original boundary cases after selection
Display the candidate zone IDs alongside the final assignment for a few boundary records. Verify all 79 cases have one output row and that each selected ID equals the smallest candidate ID. Candidate IDs are sorted only for readable display; the assignment itself was made by the Window above.


In [19]:
boundary_choices = traffic_zone_candidates.groupBy(*traffic_key).agg(
    F.count("zone_id").alias("candidate_count"),
    F.sort_array(F.collect_set("zone_id")).alias("candidate_zone_ids"),
    F.min("zone_id").alias("expected_zone_id"),
).filter(F.col("candidate_count") > 1)

boundary_enriched = traffic_enriched_df.join(boundary_choices, traffic_key, "inner")
boundary_case_count = boundary_choices.count()
boundary_output_count = boundary_enriched.count()
incorrect_boundary_choices = boundary_enriched.filter(~F.col("zone_id").eqNullSafe(F.col("expected_zone_id"))).count()

print("Original boundary cases:", boundary_case_count)
print("Boundary records after enrichment:", boundary_output_count)
print("Assignments differing from smallest zone_id:", incorrect_boundary_choices)
assert boundary_case_count == boundary_output_count == 79
assert incorrect_boundary_choices == 0

boundary_enriched.select(
    *traffic_key, "location_lat", "location_lon", "candidate_zone_ids", "zone_id", "zone_name", "zone_type"
).orderBy(*traffic_key).show(5, truncate=False)


Original boundary cases: 79
Boundary records after enrichment: 79
Assignments differing from smallest zone_id: 0


+---------+-------------------+------------+------------+------------------------+----------+----------------------+-----------+
|sensor_id|timestamp          |location_lat|location_lon|candidate_zone_ids      |zone_id   |zone_name             |zone_type  |
+---------+-------------------+------------+------------+------------------------+----------+----------------------+-----------+
|TRF-0019 |2025-01-21 21:30:00|40.706845   |-73.934579  |[ZONE-6019, ZONE-6020]  |ZONE-6019 |Residential Zone 6019 |residential|
|TRF-0025 |2025-02-22 04:00:00|40.747368   |-74.036688  |[ZONE-15025, ZONE-15215]|ZONE-15025|Residential Zone 15025|residential|
|TRF-0026 |2025-02-11 18:05:00|40.733895   |-74.000017  |[ZONE-12026, ZONE-12216]|ZONE-12026|Commercial Zone 12026 |commercial |
|TRF-0042 |2025-01-01 03:25:00|40.680842   |-74.012987  |[ZONE-0042, ZONE-0232]  |ZONE-0042 |Campus Zone 0042      |campus     |
|TRF-0104 |2025-03-25 16:35:00|40.786947   |-73.903566  |[ZONE-24104, ZONE-24294]|ZONE-24104|Comm

# Additional EDA and Enrichment Insights
These additional checks reuse the completed DataFrames and leave the mapping rule unchanged. They examine what the labels mean, how categories differ, and whether zone-level averages have enough observations to interpret. All operations are read-only; no processed dataset is saved.

## 1. Understand City Zone names
A label is not automatically a real place name. Inspect the actual strings, count distinct names, and count IDs per name before interpreting them as geographic groups. A type label followed by an identifier-like number is evidence of a naming pattern, not proof of data provenance.


In [20]:
from pyspark.sql import functions as F

city_zones_df.select("zone_id", "zone_name", "zone_type", "lat_min", "lat_max", "lon_min", "lon_max", "population").orderBy("zone_id").show(10, truncate=False)
zone_name_counts = city_zones_df.groupBy("zone_name").agg(F.countDistinct("zone_id").alias("zone_ids_per_name"))
zone_name_stats = city_zones_df.agg(
    F.count("*").alias("rows"),
    F.countDistinct("zone_id").alias("distinct_ids"),
    F.countDistinct("zone_name").alias("distinct_names"),
    F.sum(F.when(F.col("zone_name").rlike(r"^[A-Za-z ]+ Zone [0-9]+$"), 1).otherwise(0)).alias("pattern_names"),
).first()
print(zone_name_stats.asDict())
print("Distinct names and their number of zone IDs (sample):")
zone_name_counts.orderBy(F.desc("zone_ids_per_name"), "zone_name").show(10, truncate=False)
print("Distribution of IDs per name:")
zone_name_counts.groupBy("zone_ids_per_name").count().orderBy("zone_ids_per_name").show(truncate=False)
print(f"{zone_name_stats['pattern_names']:,} of {zone_name_stats['rows']:,} labels follow a words + 'Zone' + number pattern.")
if zone_name_stats['pattern_names'] == zone_name_stats['rows']:
    print("The names appear synthetic/generated from category labels and numbers; they do not establish real geographic names or neighborhoods.")
print(f"There are {zone_name_stats['distinct_names']:,} distinct names for {zone_name_stats['distinct_ids']:,} zone IDs; inspect the IDs-per-name distribution for repeated area labels.")


+---------+---------------------+-----------+-------+---------+----------+----------+----------+
|zone_id  |zone_name            |zone_type  |lat_min|lat_max  |lon_min   |lon_max   |population|
+---------+---------------------+-----------+-------+---------+----------+----------+----------+
|ZONE-0001|Residential Zone 0001|residential|40.68  |40.680842|-74.05    |-74.049105|3824      |
|ZONE-0002|Commercial Zone 0002 |commercial |40.68  |40.680842|-74.049105|-74.048211|602       |
|ZONE-0003|Industrial Zone 0003 |industrial |40.68  |40.680842|-74.048211|-74.047316|663       |
|ZONE-0004|Mixed Use Zone 0004  |mixed_use  |40.68  |40.680842|-74.047316|-74.046421|3506      |
|ZONE-0005|Park Zone 0005       |park       |40.68  |40.680842|-74.046421|-74.045526|57        |
|ZONE-0006|Campus Zone 0006     |campus     |40.68  |40.680842|-74.045526|-74.044632|1943      |
|ZONE-0007|Residential Zone 0007|residential|40.68  |40.680842|-74.044632|-74.043737|3679      |
|ZONE-0008|Commercial Zone 000

{'rows': 36000, 'distinct_ids': 36000, 'distinct_names': 36000, 'pattern_names': 36000}
Distinct names and their number of zone IDs (sample):


+----------------+-----------------+
|zone_name       |zone_ids_per_name|
+----------------+-----------------+
|Campus Zone 0006|1                |
|Campus Zone 0012|1                |
|Campus Zone 0018|1                |
|Campus Zone 0024|1                |
|Campus Zone 0030|1                |
|Campus Zone 0036|1                |
|Campus Zone 0042|1                |
|Campus Zone 0048|1                |
|Campus Zone 0054|1                |
|Campus Zone 0060|1                |
+----------------+-----------------+
only showing top 10 rows
Distribution of IDs per name:


+-----------------+-----+
|zone_ids_per_name|count|
+-----------------+-----+
|1                |36000|
+-----------------+-----+

36,000 of 36,000 labels follow a words + 'Zone' + number pattern.
The names appear synthetic/generated from category labels and numbers; they do not establish real geographic names or neighborhoods.
There are 36,000 distinct names for 36,000 zone IDs; inspect the IDs-per-name distribution for repeated area labels.


## 2. City Zone type distribution and population
Count zones by type, divide by the total for percentages, and summarize population. These describe reference-table rows, not Traffic observation counts or a verified real-world population census.


In [21]:
zone_type_distribution = city_zones_df.groupBy("zone_type").agg(
    F.count("*").alias("zone_count"),
    F.avg("population").alias("avg_population"),
    F.min("population").alias("min_population"),
    F.max("population").alias("max_population"),
).withColumn("percent_of_zones", F.round(F.col("zone_count") / F.lit(zone_name_stats["rows"]) * 100, 2)).orderBy(F.desc("zone_count"), "zone_type")
zone_type_distribution.show(truncate=False)
zone_type_rows = zone_type_distribution.collect()  # Only the small category summary.
print("Zone counts by type:", {r.zone_type: r.zone_count for r in zone_type_rows})
print("Highest mean population:", max(zone_type_rows, key=lambda r: r.avg_population).zone_type)
print("Lowest mean population:", min(zone_type_rows, key=lambda r: r.avg_population).zone_type)


+-----------+----------+------------------+--------------+--------------+----------------+
|zone_type  |zone_count|avg_population    |min_population|max_population|percent_of_zones|
+-----------+----------+------------------+--------------+--------------+----------------+
|campus     |6000      |3388.327333333333 |800           |6000          |16.67           |
|commercial |6000      |2233.775166666667 |500           |4000          |16.67           |
|industrial |6000      |650.0631666666667 |100           |1200          |16.67           |
|mixed_use  |6000      |4714.5886666666665|1500          |7999          |16.67           |
|park       |6000      |124.77966666666667|0             |250           |16.67           |
|residential|6000      |7020.4875         |2001          |12000         |16.67           |
+-----------+----------+------------------+--------------+--------------+----------------+



Zone counts by type: {'campus': 6000, 'commercial': 6000, 'industrial': 6000, 'mixed_use': 6000, 'park': 6000, 'residential': 6000}
Highest mean population: residential
Lowest mean population: park


## 3. Traffic categories
Group by road type and congestion level to compare observation counts, mean vehicle counts, and mean observation speeds. Differences are descriptive associations; these summaries do not establish causes.


In [22]:
traffic_by_road_type = traffic_df.groupBy("road_type").agg(
    F.count("*").alias("observations"), F.avg("vehicle_count").alias("avg_vehicle_count"), F.avg("avg_speed").alias("avg_speed")
).orderBy(F.desc("observations"), "road_type")
traffic_by_congestion = traffic_df.groupBy("congestion_level").agg(
    F.count("*").alias("observations"), F.avg("vehicle_count").alias("avg_vehicle_count"), F.avg("avg_speed").alias("avg_speed")
).orderBy(F.desc("observations"), "congestion_level")
traffic_by_road_type.show(truncate=False)
traffic_by_congestion.show(truncate=False)
road_rows = traffic_by_road_type.collect()
congestion_rows = traffic_by_congestion.collect()
for label, rows, category in [("Road type", road_rows, "road_type"), ("Congestion", congestion_rows, "congestion_level")]:
    for metric in ("avg_vehicle_count", "avg_speed"):
        high = max(rows, key=lambda r: r[metric])
        low = min(rows, key=lambda r: r[metric])
        print(f"{label}: highest {metric} = {high[category]} ({high[metric]:.2f}); lowest = {low[category]} ({low[metric]:.2f}).")


+-----------+------------+-----------------+------------------+
|road_type  |observations|avg_vehicle_count|avg_speed         |
+-----------+------------+-----------------+------------------+
|downtown   |7269        |76.50598431696244|11.123309946347492|
|residential|7245        |76.10393374741201|13.898143547273937|
|highway    |7201        |76.39550062491321|36.115601999722145|
|arterial   |7164        |76.17057509771078|25.00011864879961 |
|school_zone|7121        |76.82347984833591|8.521088330290683 |
+-----------+------------+-----------------+------------------+



+----------------+------------+------------------+------------------+
|congestion_level|observations|avg_vehicle_count |avg_speed         |
+----------------+------------+------------------+------------------+
|medium          |13852       |69.30912503609586 |19.342881894311244|
|high            |12789       |113.26045820627101|12.518936586128703|
|low             |9359        |36.52174377604445 |27.070528902660524|
+----------------+------------+------------------+------------------+



Road type: highest avg_vehicle_count = school_zone (76.82); lowest = residential (76.10).
Road type: highest avg_speed = highway (36.12); lowest = school_zone (8.52).
Congestion: highest avg_vehicle_count = high (113.26); lowest = low (36.52).
Congestion: highest avg_speed = low (27.07); lowest = high (12.52).


## 4. What zone enrichment adds
The attached `zone_type` lets us compare Traffic observations across reference categories. For each type, show observation counts, vehicle/speed averages, and the percentage of observations in each congestion level. Percentage-point differences are descriptive; no statistical significance or causation is assumed.


In [23]:
traffic_by_zone_type = traffic_enriched_df.groupBy("zone_type").agg(
    F.count("*").alias("observations"),
    F.avg("vehicle_count").alias("avg_vehicle_count"),
    F.avg("avg_speed").alias("avg_speed"),
    *[F.sum(F.when(F.col("congestion_level") == level, 1).otherwise(0)).alias(level + "_count") for level in ("low", "medium", "high")],
)
for level in ("low", "medium", "high"):
    traffic_by_zone_type = traffic_by_zone_type.withColumn(level + "_percent", F.round(F.col(level + "_count") / F.col("observations") * 100, 2))
traffic_by_zone_type = traffic_by_zone_type.orderBy(F.desc("avg_vehicle_count"), "zone_type")
traffic_by_zone_type.show(truncate=False)
zone_traffic_rows = traffic_by_zone_type.collect()
for metric in ("avg_vehicle_count", "avg_speed"):
    high = max(zone_traffic_rows, key=lambda r: r[metric])
    low = min(zone_traffic_rows, key=lambda r: r[metric])
    print(f"Highest {metric}: {high.zone_type} ({high[metric]:.2f}); lowest: {low.zone_type} ({low[metric]:.2f}).")
print(f"High-congestion percentage spans {min(r.high_percent for r in zone_traffic_rows):.2f}% to {max(r.high_percent for r in zone_traffic_rows):.2f}%. Interpret this as a descriptive difference, not evidence of causation or significance.")


+-----------+------------+-----------------+------------------+---------+------------+----------+-----------+--------------+------------+
|zone_type  |observations|avg_vehicle_count|avg_speed         |low_count|medium_count|high_count|low_percent|medium_percent|high_percent|
+-----------+------------+-----------------+------------------+---------+------------+----------+-----------+--------------+------------+
|park       |5997        |77.01450725362682|18.83553776888445 |1554     |2239        |2204      |25.91      |37.34         |36.75       |
|mixed_use  |5994        |76.61344678011345|18.82980480480476 |1552     |2308        |2134      |25.89      |38.51         |35.6        |
|industrial |6005        |76.4014987510408 |18.96867277268946 |1530     |2343        |2132      |25.48      |39.02         |35.5        |
|campus     |6005        |76.2872606161532 |19.184274771024146|1563     |2322        |2120      |26.03      |38.67         |35.3        |
|residential|5997        |76.23695

Highest avg_vehicle_count: park (77.01); lowest: commercial (75.84).
Highest avg_speed: campus (19.18); lowest: residential (18.81).
High-congestion percentage spans 34.33% to 36.75%. Interpret this as a descriptive difference, not evidence of causation or significance.


## 5. Is a zone-name ranking informative?
The observed distribution below contains only one or two observations per name. We therefore do not lower the eligibility threshold to make a ranking possible: require at least **five observations** to exclude these extremely sparse groups. Five is a modest descriptive screen, not a statistical guarantee. No name qualifies in this run. Display the count distribution and a coverage ranking, but avoid interpreting averages as stable geographic patterns. The names are unique numbered labels, not repeated neighborhood groups.


In [24]:
traffic_by_zone_name = traffic_enriched_df.groupBy("zone_name").agg(
    F.count("*").alias("observations"),
    F.avg("vehicle_count").alias("avg_vehicle_count"),
    F.avg("avg_speed").alias("avg_speed"),
)
print("Observation-count distribution across names:")
traffic_by_zone_name.groupBy("observations").count().orderBy("observations").show(truncate=False)
name_observation_stats = traffic_by_zone_name.agg(F.count("*").alias("observed_names"), F.min("observations").alias("min_observations"), F.max("observations").alias("max_observations")).first()
print(name_observation_stats.asDict())
minimum_name_observations = 5
eligible_zone_names = traffic_by_zone_name.filter(F.col("observations") >= minimum_name_observations)
eligible_name_count = eligible_zone_names.count()
print(f"Names meeting the minimum of {minimum_name_observations} observations: {eligible_name_count}")
print("Most-observed names (coverage ranking, not a reliable ranking of average traffic):")
traffic_by_zone_name.orderBy(F.desc("observations"), "zone_name").show(10, truncate=False)
if eligible_name_count:
    for metric in ("avg_vehicle_count", "avg_speed", "observations"):
        print("Eligible names ranked by", metric)
        eligible_zone_names.orderBy(F.desc(metric), "zone_name").show(10, truncate=False)
else:
    print(f"Actual maximum is only {name_observation_stats['max_observations']} observations per name. No name reaches five; average-based rankings would be driven by too few observations. Numbered zone labels also do not establish meaningful neighborhood groups.")


Observation-count distribution across names:


+------------+-----+
|observations|count|
+------------+-----+
|1           |35924|
|2           |38   |
+------------+-----+



{'observed_names': 35962, 'min_observations': 1, 'max_observations': 2}


Names meeting the minimum of 5 observations: 0
Most-observed names (coverage ranking, not a reliable ranking of average traffic):


+-----------------+------------+-----------------+------------------+
|zone_name        |observations|avg_vehicle_count|avg_speed         |
+-----------------+------------+-----------------+------------------+
|Campus Zone 0288 |2           |61.0             |22.325            |
|Campus Zone 10068|2           |50.5             |24.22             |
|Campus Zone 12114|2           |48.5             |14.754999999999999|
|Campus Zone 1392 |2           |47.0             |13.71             |
|Campus Zone 26292|2           |52.0             |18.045            |
|Campus Zone 3054 |2           |67.0             |26.515            |
|Campus Zone 30648|2           |73.5             |33.379999999999995|
|Campus Zone 32172|2           |95.5             |7.0               |
|Campus Zone 35922|2           |147.0            |15.045            |
|Campus Zone 4698 |2           |101.0            |11.425            |
+-----------------+------------+-----------------+------------------+
only showing top 10 

## 6. Final EDA summary

### City Zones: labels, categories, and population
All **36,000 zone names are distinct**, with **one zone ID per name**. All follow a words + `Zone` + number pattern (for example, `Residential Zone 0001`). They appear synthetic/generated; these strings do not establish real geographic names, boroughs, or neighborhoods. Provenance cannot be proved from the naming pattern alone.

Each of the six types has **6,000 zones (16.67% rounded)**. Rounded percentages sum slightly above 100% because each exact share is one-sixth.

| Zone type | Zones | Mean population | Minimum | Maximum |
|---|---:|---:|---:|---:|
| campus | 6,000 | 3,388.33 | 800 | 6,000 |
| commercial | 6,000 | 2,233.78 | 500 | 4,000 |
| industrial | 6,000 | 650.06 | 100 | 1,200 |
| mixed_use | 6,000 | 4,714.59 | 1,500 | 7,999 |
| park | 6,000 | 124.78 | 0 | 250 |
| residential | 6,000 | 7,020.49 | 2,001 | 12,000 |

Residential has the highest mean population and park the lowest. These are reference-data values, not an independently verified census. No population-to-traffic causal relationship was tested.

### Traffic: road and congestion categories

| Road type | Observations | Mean vehicle count | Mean speed |
|---|---:|---:|---:|
| downtown | 7,269 | 76.51 | 11.12 |
| residential | 7,245 | 76.10 | 13.90 |
| highway | 7,201 | 76.40 | 36.12 |
| arterial | 7,164 | 76.17 | 25.00 |
| school_zone | 7,121 | 76.82 | 8.52 |

Mean vehicle counts are close across road types (76.10–76.82), while mean speeds span 8.52–36.12 in source-data units. Highway is fastest on average and school_zone slowest. These comparisons do not explain the causes.

| Congestion level | Observations | Mean vehicle count | Mean speed |
|---|---:|---:|---:|
| medium | 13,852 | 69.31 | 19.34 |
| high | 12,789 | 113.26 | 12.52 |
| low | 9,359 | 36.52 | 27.07 |

Medium is the most frequent category. High congestion is associated with higher vehicle counts and lower speeds than low congestion; this is a descriptive association.

### Enriched Traffic: comparisons enabled by zone information

| Zone type | Traffic observations | Mean vehicle count | Mean speed | High congestion (%) |
|---|---:|---:|---:|---:|
| park | 5,997 | 77.01 | 18.84 | 36.75 |
| mixed_use | 5,994 | 76.61 | 18.83 | 35.60 |
| industrial | 6,005 | 76.40 | 18.97 | 35.50 |
| campus | 6,005 | 76.29 | 19.18 | 35.30 |
| residential | 5,997 | 76.24 | 18.81 | 34.33 |
| commercial | 6,002 | 75.84 | 18.94 | 35.65 |

Attaching zone metadata enables comparisons by zone type that were unavailable in the original Traffic columns. Park has the highest mean vehicle count (77.01), commercial the lowest (75.84). Campus has the highest mean speed (19.18), residential the lowest (18.81). High-congestion shares span **34.33%–36.75%**, a **2.42 percentage-point** spread using rounded values. Differences are modest descriptively; no significance test, practical-effect threshold, or causal analysis was performed. Road type and zone type are different attributes and should not be conflated.

### Name-level limitations and points to investigate before the PR
There are **35,962 observed zone names**: **35,924 have one observation** and **38 have two**. Therefore **38 of the 36,000 reference names have no assigned Traffic observation**. This does not mean any Traffic record is unmatched: coverage of zones and coverage of Traffic records are different questions.

No name reaches the minimum of five observations. Numbered labels and extremely sparse per-name coverage make average-based zone-name rankings uninformative here; only a small coverage ranking is shown. Before presenting geographic insights, clarify the provenance and intended meaning of the generated-looking labels and why per-zone coverage is so sparse. The perfectly balanced zone-type counts and category-specific population ranges are also useful provenance questions, not proof of a generation process. No new mapping rule or source-data modification was introduced.

### Previously verified mapping results retained

| Check | Result |
|---|---:|
| Original Traffic records | 36,000 |
| Exactly one candidate zone | 35,921 |
| Zero candidate zones | 0 |
| Multiple candidate zones | 79 |
| Inclusive candidate rows | 36,079 |
| Final enriched rows | 36,000 |
| Null zone assignments | 0 |
| Duplicate Traffic logical keys | 0 |

These are the completed mapping checks recorded above, not a new mapping implementation. The smallest-zone-ID boundary rule remains unchanged. This section adds read-only EDA only; no processed output has been saved.


# Temporal Quality and Statistical Assessment
This section extends the completed EDA without rerunning category analyses or changing source data, validation rules, or mapping. Temporal calculations use the existing Spark session time zone. Elapsed intervals use epoch seconds, so clock changes are not mistaken for missing observations.

## 1. Traffic temporal coverage
Derive calendar features in a new DataFrame. Calendar coverage describes the dataset as a whole; counts per sensor reveal how much repeated observation each sensor actually provides.


In [25]:
from pyspark.sql import functions as F
from pyspark.sql import Window

traffic_temporal = (traffic_df
    .withColumn("event_date", F.to_date("timestamp"))
    .withColumn("hour_of_day", F.hour("timestamp"))
    .withColumn("day_of_week", F.dayofweek("timestamp"))
    .withColumn("week_of_year", F.weekofyear("timestamp"))
    .withColumn("week_start", F.to_date(F.date_trunc("week", "timestamp"))))
temporal_coverage = traffic_temporal.agg(
    F.min("timestamp").cast("string").alias("min_timestamp"),
    F.max("timestamp").cast("string").alias("max_timestamp"),
    (F.max(F.col("timestamp").cast("long")) - F.min(F.col("timestamp").cast("long"))).alias("elapsed_seconds"),
    F.countDistinct("event_date").alias("calendar_dates"),
    F.countDistinct("sensor_id").alias("sensors"),
    F.count("*").alias("observations"),
    F.countDistinct("day_of_week").alias("weekdays"),
    F.countDistinct("week_start").alias("calendar_weeks"),
).first()
print("Session time zone:", spark.conf.get("spark.sql.session.timeZone"))
print(temporal_coverage.asDict())
print("Elapsed days:", temporal_coverage.elapsed_seconds / 86400)
observations_per_sensor = traffic_df.groupBy("sensor_id").count()
sensor_observation_stats = observations_per_sensor.agg(F.min("count").alias("minimum"), F.avg("count").alias("average"), F.max("count").alias("maximum")).first()
print("Observations per sensor:", sensor_observation_stats.asDict())
print(f"The dataset covers {temporal_coverage.calendar_dates} calendar dates across {temporal_coverage.sensors} sensors; this does not imply dense coverage for each sensor.")


Session time zone: America/New_York
{'min_timestamp': '2025-01-01 00:00:00', 'max_timestamp': '2025-05-05 23:55:00', 'elapsed_seconds': 10796100, 'calendar_dates': 125, 'sensors': 3000, 'observations': 36000, 'weekdays': 7, 'calendar_weeks': 19}
Elapsed days: 124.95486111111111


Observations per sensor: {'minimum': 12, 'average': 12.0, 'maximum': 12}
The dataset covers 125 calendar dates across 3000 sensors; this does not imply dense coverage for each sensor.


## 2. Per-sensor intervals and gaps
`lag()` retrieves the previous timestamp within a Window partitioned by sensor and ordered by timestamp. Subtract epoch seconds to measure elapsed time between observations of the same sensor. The first observation per sensor has no previous interval and is excluded.

Infer the typical interval from the most frequent observed duration(s). Flag durations exceeding **1.5 times the largest modal duration** as unusually large for exploratory screening. This explicit heuristic is not a contractual reporting schedule or proof of missing records. Display all tied modes, the interval distribution, and non-modal intervals. No missing observations are manufactured.


In [26]:
sensor_time_window = Window.partitionBy("sensor_id").orderBy("timestamp")
traffic_intervals = (traffic_df.select("sensor_id", "timestamp")
    .withColumn("previous_timestamp", F.lag("timestamp").over(sensor_time_window))
    .withColumn("interval_seconds", F.col("timestamp").cast("long") - F.col("previous_timestamp").cast("long"))
    .filter(F.col("previous_timestamp").isNotNull()))
interval_distribution = traffic_intervals.groupBy("interval_seconds").count().orderBy(F.desc("count"), "interval_seconds")
interval_distribution.show(30, truncate=False)
mode_frequency = interval_distribution.agg(F.max("count")).first()[0]
interval_modes = interval_distribution.filter(F.col("count") == mode_frequency)
print("Most common interval(s):")
interval_modes.show(truncate=False)
typical_interval_seconds = interval_modes.agg(F.max("interval_seconds")).first()[0]
gap_threshold_seconds = typical_interval_seconds * 1.5
interval_stats = traffic_intervals.agg(
    F.count("*").alias("intervals"), F.min("interval_seconds").alias("minimum_seconds"),
    F.max("interval_seconds").alias("maximum_seconds"),
    F.sum(F.when(F.col("interval_seconds") > gap_threshold_seconds, 1).otherwise(0)).alias("large_gaps"),
).first()
nonmodal_intervals = traffic_intervals.join(interval_modes.select("interval_seconds"), "interval_seconds", "left_anti").count()
print("Interval statistics:", interval_stats.asDict())
print("Non-modal intervals:", nonmodal_intervals)
print("Large-gap threshold (seconds):", gap_threshold_seconds)
print("Global consecutive timestamps may belong to different sensors; they do not define a per-sensor reporting schedule.")
print("Per-sensor elapsed spacing is regular in the observed range." if nonmodal_intervals == 0 else "Per-sensor elapsed spacing varies; inspect the distribution and gap screen.")


+----------------+-----+
|interval_seconds|count|
+----------------+-----+
|900000          |30000|
|896400          |3000 |
+----------------+-----+



Most common interval(s):


+----------------+-----+
|interval_seconds|count|
+----------------+-----+
|900000          |30000|
+----------------+-----+



Interval statistics: {'intervals': 33000, 'minimum_seconds': 896400, 'maximum_seconds': 900000, 'large_gaps': 0}
Non-modal intervals: 3000
Large-gap threshold (seconds): 1350000.0
Global consecutive timestamps may belong to different sensors; they do not define a per-sensor reporting schedule.
Per-sensor elapsed spacing varies; inspect the distribution and gap screen.


## 3. Daily and weekly comparisons
Daily summaries are created when at least two dates exist, day-of-week summaries when several weekdays exist, and weekly summaries when at least three calendar weeks exist. These thresholds permit descriptive comparison, not a claim of seasonality. Weekly groups use Monday week-start dates to avoid mixing the same week number across years. Show date coverage and observation counts because endpoint days/weeks can be partial. Weekday numbers follow Spark: Sunday = 1 through Saturday = 7.


In [27]:
if temporal_coverage.calendar_dates >= 2:
    daily_traffic = traffic_temporal.groupBy("event_date").agg(F.count("*").alias("observations"), F.avg("vehicle_count").alias("avg_vehicle_count"), F.avg("avg_speed").alias("avg_speed")).orderBy("event_date")
    daily_traffic.show(temporal_coverage.calendar_dates, truncate=False)
    print("Daily count range:")
    daily_traffic.agg(F.min("observations"), F.max("observations")).show()
else:
    print("Insufficient dates for daily comparisons.")
if temporal_coverage.weekdays > 1:
    weekday_traffic = traffic_temporal.groupBy("day_of_week").agg(F.count("*").alias("observations"), F.countDistinct("event_date").alias("dates"), F.avg("vehicle_count").alias("avg_vehicle_count"), F.avg("avg_speed").alias("avg_speed")).orderBy("day_of_week")
    weekday_traffic.show(7, truncate=False)
if temporal_coverage.calendar_weeks >= 3:
    weekly_traffic = traffic_temporal.groupBy("week_start").agg(F.count("*").alias("observations"), F.countDistinct("event_date").alias("dates"), F.avg("vehicle_count").alias("avg_vehicle_count"), F.avg("avg_speed").alias("avg_speed")).orderBy("week_start")
    weekly_traffic.show(temporal_coverage.calendar_weeks, truncate=False)
    print("Weekly comparisons are descriptive; account for partial weeks. This does not establish seasonality.")
else:
    print("Fewer than three calendar weeks: insufficient coverage for the chosen weekly comparison.")


+----------+------------+------------------+------------------+
|event_date|observations|avg_vehicle_count |avg_speed         |
+----------+------------+------------------+------------------+
|2025-01-01|288         |79.40277777777777 |17.952604166666664|
|2025-01-02|288         |79.71875          |17.93329861111111 |
|2025-01-03|288         |79.59722222222223 |18.22086805555556 |
|2025-01-04|288         |77.67013888888889 |18.06208333333332 |
|2025-01-05|288         |80.03472222222223 |17.39142361111113 |
|2025-01-06|288         |80.49652777777777 |18.646215277777785|
|2025-01-07|288         |77.62847222222223 |18.223749999999995|
|2025-01-08|288         |82.52777777777777 |17.645729166666666|
|2025-01-09|288         |77.71180555555556 |18.950173611111094|
|2025-01-10|288         |57.94444444444444 |21.340729166666666|
|2025-01-11|288         |76.45833333333333 |19.364027777777782|
|2025-01-12|288         |79.35416666666667 |17.22645833333334 |
|2025-01-13|288         |80.048611111111

+-----------------+-----------------+
|min(observations)|max(observations)|
+-----------------+-----------------+
|              288|              288|
+-----------------+-----------------+



+-----------+------------+-----+-----------------+------------------+
|day_of_week|observations|dates|avg_vehicle_count|avg_speed         |
+-----------+------------+-----+-----------------+------------------+
|1          |5184        |18   |75.51909722222223|18.831510416666667|
|2          |5184        |18   |75.15837191358025|18.94799961419753 |
|3          |4896        |17   |78.71323529411765|18.67754697712418 |
|4          |5184        |18   |77.40027006172839|18.63927662037037 |
|5          |5184        |18   |73.66550925925925|19.428759645061728|
|6          |5184        |18   |76.66898148148148|19.08424575617284 |
|7          |5184        |18   |77.79629629629629|18.870308641975303|
+-----------+------------+-----+-----------------+------------------+



+----------+------------+-----+-----------------+------------------+
|week_start|observations|dates|avg_vehicle_count|avg_speed         |
+----------+------------+-----+-----------------+------------------+
|2024-12-30|1440        |5    |79.28472222222223|17.912055555555558|
|2025-01-06|2016        |7    |76.01736111111111|18.771011904761902|
|2025-01-13|2016        |7    |78.35813492063492|18.41673115079365 |
|2025-01-20|2016        |7    |74.19097222222223|19.4269246031746  |
|2025-01-27|2016        |7    |75.67311507936508|18.709945436507937|
|2025-02-03|2016        |7    |76.19940476190476|18.695372023809522|
|2025-02-10|2016        |7    |75.56498015873017|19.191145833333337|
|2025-02-17|2016        |7    |74.70287698412699|19.25345734126984 |
|2025-02-24|2016        |7    |78.15178571428571|18.608253968253965|
|2025-03-03|2016        |7    |77.76190476190476|18.638070436507938|
|2025-03-10|2016        |7    |74.39484126984127|19.43246527777778 |
|2025-03-17|2016        |7    |75.

### Investigate the one-hour interval difference
Compare local wall-clock durations and UTC offsets with elapsed seconds. This distinguishes clock changes from unusually long reporting intervals.


In [28]:
# Check whether the one-hour variation corresponds to a UTC-offset change.
interval_clock_check = traffic_intervals.withColumn(
    "wall_clock_seconds", F.expr("timestampdiff(SECOND, cast(previous_timestamp as timestamp_ntz), cast(timestamp as timestamp_ntz))")
).withColumn("previous_offset", F.date_format("previous_timestamp", "XXX")).withColumn("current_offset", F.date_format("timestamp", "XXX"))
interval_clock_check.groupBy("interval_seconds", "wall_clock_seconds", "previous_offset", "current_offset").count().orderBy("interval_seconds", "previous_offset").show(truncate=False)
print("Daily observation-count range (includes clock-change effects):")
daily_traffic.agg(F.min("observations"), F.max("observations")).show()


+----------------+------------------+---------------+--------------+-----+
|interval_seconds|wall_clock_seconds|previous_offset|current_offset|count|
+----------------+------------------+---------------+--------------+-----+
|896400          |896400            |-04:00         |-04:00        |12   |
|896400          |900000            |-05:00         |-04:00        |2988 |
|900000          |900000            |-04:00         |-04:00        |13668|
|900000          |903600            |-05:00         |-04:00        |12   |
|900000          |900000            |-05:00         |-05:00        |16320|
+----------------+------------------+---------------+--------------+-----+

Daily observation-count range (includes clock-change effects):


+-----------------+-----------------+
|min(observations)|max(observations)|
+-----------------+-----------------+
|              288|              288|
+-----------------+-----------------+



## 4. Exploratory IQR outlier assessment
Business/range validation asks whether a value violates an expected rule. IQR analysis asks whether a value is unusual relative to the observed distribution. A statistical outlier is not automatically bad or invalid data.

For each field, calculate exact interpolated quartiles using Spark `percentile`, then IQR = Q3 − Q1. Flag values strictly outside Q1 − 1.5 × IQR and Q3 + 1.5 × IQR. This global distribution screen does not adjust for road type or hour. It supplements the completed shared validator; no values are removed, capped, or replaced.


In [29]:
iqr_results = []
for field in ("vehicle_count", "avg_speed"):
    stats = traffic_df.agg(F.expr(f"percentile({field}, array(0.25, 0.75))").alias("quartiles"), F.min(field).alias("minimum"), F.max(field).alias("maximum")).first()
    q1, q3 = stats.quartiles
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    outside = traffic_df.filter((F.col(field) < lower) | (F.col(field) > upper)).count()
    iqr_results.append((field, float(q1), float(q3), float(iqr), float(lower), float(upper), float(stats.minimum), float(stats.maximum), outside, outside / temporal_coverage.observations * 100))
traffic_iqr_assessment = spark.createDataFrame(iqr_results, ["field", "q1", "q3", "iqr", "lower_boundary", "upper_boundary", "observed_min", "observed_max", "outlier_count", "outlier_percent"])
traffic_iqr_assessment.show(truncate=False)


+-------------+----+-----+-----+--------------+--------------+------------+------------+-------------+------------------+
|field        |q1  |q3   |iqr  |lower_boundary|upper_boundary|observed_min|observed_max|outlier_count|outlier_percent   |
+-------------+----+-----+-----+--------------+--------------+------------+------------+-------------+------------------+
|vehicle_count|47.0|101.0|54.0 |-34.0         |182.0         |15.0        |186.0       |139          |0.3861111111111111|
|avg_speed    |9.55|24.76|15.21|-13.265       |47.575        |5.0         |61.13       |1498         |4.161111111111111 |
+-------------+----+-----+-----+--------------+--------------+------------+------------+-------------+------------------+



## 5. Temporal feature demonstration and preservation check
`event_date`, `hour_of_day`, `day_of_week`, and `week_of_year` are derived from timestamp to support grouping and downstream analytics. They are not predictive ML features. They live in `traffic_temporal`; original Traffic columns and enrichment logic remain unchanged.


In [ ]:
traffic_temporal.select("sensor_id", "timestamp", "event_date", "hour_of_day", "day_of_week", "week_of_year").orderBy("timestamp", "sensor_id").show(5, truncate=False)
final_source_count = traffic_df.count()
final_enriched_count = traffic_enriched_df.count()
print("Traffic rows after temporal assessment:", final_source_count)
print("Enriched Traffic rows after temporal assessment:", final_enriched_count)
assert final_source_count == final_enriched_count == 36000


+---------+-------------------+----------+-----------+-----------+------------+
|sensor_id|timestamp          |event_date|hour_of_day|day_of_week|week_of_year|
+---------+-------------------+----------+-----------+-----------+------------+
|TRF-0001 |2025-01-01 00:00:00|2025-01-01|0          |4          |1           |
|TRF-0002 |2025-01-01 00:05:00|2025-01-01|0          |4          |1           |
|TRF-0003 |2025-01-01 00:10:00|2025-01-01|0          |4          |1           |
|TRF-0004 |2025-01-01 00:15:00|2025-01-01|0          |4          |1           |
|TRF-0005 |2025-01-01 00:20:00|2025-01-01|0          |4          |1           |
+---------+-------------------+----------+-----------+-----------+------------+
only showing top 5 rows


Traffic rows after temporal assessment: 36000
Enriched Traffic rows after temporal assessment: 36000


26/09/13 23:06:02 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 253301 ms exceeds timeout 120000 ms
26/09/13 23:06:02 WARN SparkContext: Killing executors is not supported by current scheduler.
26/09/13 23:23:08 WARN Executor: Issue communicating with driver in heartbeater
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:70)
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:44)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:101)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:85)
	at org.apache.spark.storage.BlockManagerMaster.registerBlockManager(BlockManagerMaster.scala:85)
	at org.apache.spark.storage.BlockManager.reregister(BlockManager.scala:707

## 6. Final Traffic quality summary

| Measure | Calculated result |
|---|---|
| Traffic observations | 36,000 |
| Distinct sensors | 3,000 |
| Earliest timestamp (America/New_York) | 2025-01-01 00:00:00 |
| Latest timestamp (America/New_York) | 2025-05-05 23:55:00 |
| Elapsed time | 10,796,100 seconds (124 days, 22 hours, 55 minutes) |
| Distinct calendar dates | 125 |
| Observations per sensor: minimum / average / maximum | 12 / 12.0 / 12 |
| Consecutive same-sensor intervals | 33,000 |
| Modal interval | 900,000 seconds = 10 days, 10 hours (30,000 intervals) |
| Minimum interval | 896,400 seconds = 10 days, 9 hours (3,000 intervals) |
| Maximum interval | 900,000 seconds |
| Unusually large intervals | 0 above the exploratory threshold of 1,350,000 seconds |
| Calendar weeks represented | 19 |
| Final enriched records | 36,000 |

There is broad dataset-level coverage but sparse per-sensor sampling. Consecutive rows in the displayed global sample are five minutes apart and belong to different sensors; this is not evidence that each sensor reports every five minutes. No missing records were inferred or generated. The large-gap test is relative to the observed mode and cannot establish compliance with an external reporting schedule or detect gaps before the first/after the last observation.

**Clock behavior:** 2,988 shortened elapsed intervals correspond to an offset change from -05:00 to -04:00 while retaining 900,000 local wall-clock seconds. Another 12 intervals have 896,400 wall-clock seconds with no offset change, and 12 have 903,600 wall-clock seconds while crossing the offset change. Thus the data is near-regular, with one-hour variations; those 24 nonstandard wall-clock intervals merit timestamp/source-conversion review. The evidence does not justify labeling these as missing observations or claiming perfect regularity.

**Daily and weekly analysis:** Daily descriptive comparison is supported by 125 dates, each with 288 observations. All seven weekdays are represented. Weekly comparison is supported by 19 calendar weeks, but the first and last are partial (5 dates/1,440 observations and 1 date/288 observations). The other 17 weeks each have 7 dates/2,016 observations. Compare means with awareness of different day coverage and changing sensor composition. This range does not establish seasonality or dense individual-sensor trends.

| IQR assessment | vehicle_count | avg_speed |
|---|---:|---:|
| Q1 | 47.00 | 9.55 |
| Q3 | 101.00 | 24.76 |
| IQR | 54.00 | 15.21 |
| Lower boundary | -34.00 | -13.265 |
| Upper boundary | 182.00 | 47.575 |
| Observed minimum | 15.00 | 5.00 |
| Observed maximum | 186.00 | 61.13 |
| Outside boundaries | 139 | 1,498 |
| Outside percentage | 0.3861% | 4.1611% |

These are statistical flags, not failed business rules. All flagged values are above the upper boundaries; observed minima exceed the lower boundaries. The exact interpolated speed Q3 here is 24.76, whereas the earlier shared validator's summary reported 24.75; different quantile methods can produce slightly different cutoffs. The shared validator was not changed or rerun, and no observations were modified.

This assessment complements the completed shared validation, hourly patterns, road-type and congestion analysis, City Zone mapping, and Traffic-by-zone-type summaries. It adds timing and distribution context without replacing those results. Final checks confirm `traffic_df` and `traffic_enriched_df` each still contain 36,000 rows.

## 7. Important dataset limitations

- Zone names appear synthetic/generated rather than real neighborhood names: 36,000 distinct names correspond to 36,000 IDs. This is evidence from naming patterns, not proof of provenance.
- Individual-name Traffic coverage is extremely sparse: 35,924 names have one observation and 38 have two. Name-level average rankings were intentionally not treated as meaningful. Zone-type aggregation has more observations per group and is more defensible for descriptive comparisons.
- The original 79 multiple-zone matches were boundary cases. Smallest-zone-ID selection resolved them deterministically; the selected zone is not claimed to be geographically superior.
- Each sensor has only 12 observations, approximately 10 days and 10 hours apart. Dataset-wide daily coverage should not be mistaken for a dense time series for each sensor.
- One-hour timestamp variations and the 24 nonstandard wall-clock intervals warrant review of source timestamp semantics. Partial weeks limit direct comparisons, and annual seasonality is not established.
- Global IQR thresholds combine different road types and times. An unusually high speed relative to the whole dataset need not be unusual for its road type; statistical flags alone do not justify deleting data.

Only this notebook was updated. The added cells perform read-only DataFrame analysis; no database writes, processed dataset output, shared-source changes, README changes, or test changes were made.
